# Lakebase Search retrieval — execution evidence

Proves the Assist step retrieves from the **Build 1 Lakebase Search index** (pgvector HNSW + tsvector GIN on `northpeak_ops.products`), not a separate vector store. Runs the hybrid RRF query live; outputs below are real results.

In [1]:
import sys, json, re, subprocess, textwrap
sys.path.insert(0, ".")
from lib import lb
conn = lb.connect("geniebandits-dev"); cur = conn.cursor()
HERO_SKU = "SKU-APP-04412"
QUERY = 'Summit Down Parka warm insulated cold weather substitute'
_STOP = {"for","the","and","with","of","to","in","on","a","an"}
def show(sql, params=None, title=None):
    if title: print(f"### {title}")
    cur.execute(sql, params) if params else cur.execute(sql)
    cols=[d[0] for d in cur.description]; rows=cur.fetchall()
    print(" | ".join(cols))
    for r in rows: print("  " + " | ".join(str(x) for x in r))
    print(f"({len(rows)} row(s))\n"); return rows

In [2]:
show("""SELECT indexname, indexdef FROM pg_indexes
        WHERE schemaname='northpeak_ops' AND tablename='products'
          AND (indexdef ILIKE '%hnsw%' OR indexdef ILIKE '%gin%')
        ORDER BY indexname""", title="Build 1 Lakebase Search indexes (vector + full-text)")

### Build 1 Lakebase Search indexes (vector + full-text)
indexname | indexdef
  products_embedding_hnsw | CREATE INDEX products_embedding_hnsw ON northpeak_ops.products USING hnsw (embedding vector_cosine_ops)
  products_search_gin | CREATE INDEX products_search_gin ON northpeak_ops.products USING gin (search_document)
(2 row(s))



[('products_embedding_hnsw',
  'CREATE INDEX products_embedding_hnsw ON northpeak_ops.products USING hnsw (embedding vector_cosine_ops)'),
 ('products_search_gin',
  'CREATE INDEX products_search_gin ON northpeak_ops.products USING gin (search_document)')]

In [3]:
# generate the query embedding via the governed FM (same path Build 1 used)
out = subprocess.check_output(["databricks","experimental","aitools","tools","query",
  f"SELECT to_json(ai_query('databricks-gte-large-en', '{QUERY}')) AS emb",
  "--profile","rkm-sandbox-1","-o","json"], text=True)
qvec = "[" + ",".join(repr(float(x)) for x in json.loads(json.loads(out)[0]["emb"])) + "]"
tsq = " | ".join(w for w in re.findall(r"[A-Za-z]+", QUERY.lower()) if len(w)>2 and w not in _STOP)
print("query:", QUERY); print("lexical tsquery:", tsq); print("embedding dims: 1024")

query: Summit Down Parka warm insulated cold weather substitute
lexical tsquery: summit | down | parka | warm | insulated | cold | weather | substitute
embedding dims: 1024


In [4]:
rrf = """
WITH ft AS (
  SELECT product_id, row_number() OVER (
           ORDER BY ts_rank_cd(search_document, to_tsquery('english', %(tsq)s)) DESC) AS rank
  FROM northpeak_ops.products
  WHERE search_document @@ to_tsquery('english', %(tsq)s) AND product_id <> %(ex)s LIMIT 50),
vec AS (
  SELECT product_id, row_number() OVER (ORDER BY embedding <=> %(qvec)s::vector) AS rank
  FROM northpeak_ops.products WHERE embedding IS NOT NULL AND product_id <> %(ex)s LIMIT 50),
fused AS (
  SELECT COALESCE(ft.product_id, vec.product_id) AS product_id, ft.rank AS ft_rank,
         vec.rank AS vec_rank,
         COALESCE(1.0/(60+ft.rank),0)+COALESCE(1.0/(60+vec.rank),0) AS rrf
  FROM ft FULL OUTER JOIN vec ON ft.product_id=vec.product_id)
SELECT p.product_id, p.product_name, p.seasonality, round(p.unit_margin::numeric,2) AS unit_margin,
       f.ft_rank, f.vec_rank, round(f.rrf::numeric,5) AS rrf_score
FROM fused f JOIN northpeak_ops.products p ON p.product_id=f.product_id
ORDER BY f.rrf DESC LIMIT 6;
"""
rows = show(rrf, {"tsq": tsq, "qvec": qvec, "ex": HERO_SKU},
            title="Hybrid retrieval (RRF of full-text + vector) — substitute candidates")
print("Retrieved from the Build 1 Lakebase Search index on northpeak_ops.products "
      "(pgvector HNSW + tsvector GIN) — not a separate vector store.")

### Hybrid retrieval (RRF of full-text + vector) — substitute candidates
product_id | product_name | seasonality | unit_margin | ft_rank | vec_rank | rrf_score
  SKU-APP-04415 | Tundra Insulated Boots | cold_weather | 124.99 | 1 | 4 | 0.03202
  SKU-APP-04413 | Alpine Fleece Jacket | cold_weather | 87.99 | 5 | 1 | 0.03178
  SKU-APP-04414 | Glacier Thermal Base Layer | cold_weather | 47.99 | 6 | 2 | 0.03128
  SKU-APP-04416 | Polar Wool Beanie | cold_weather | 26.99 | 7 | 3 | 0.03080
  SKU-APP-10001 | Apparel Item 2 | warm_weather | 41.56 | 9 | 35 | 0.02502
  SKU-APP-10000 | Apparel Item 1 | warm_weather | 38.52 | 8 | 47 | 0.02405
(6 row(s))

Retrieved from the Build 1 Lakebase Search index on northpeak_ops.products (pgvector HNSW + tsvector GIN) — not a separate vector store.
